In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scipy.stats as stats
from scipy.stats import spearmanr, hypergeom

from statsmodels.stats.multitest import multipletests

# GO enrichment
import gseapy as gp

## CellOracle
import celloracle as co

In [3]:
## DATA (CellOracle object before fitting GRN)
oracle = co.load_hdf5("../data/celloracle_data/celloracle_unfit.celloracle.oracle")

## FIT GRN — RIDGE REGRESSION PER CLUSTER

Use Links object to store raw GRNs per cluster (with weights and associated p-values):

In [4]:
# Step 1: infer GRN per cluster with Ridge Regression
# alpha: regularization strength. Higher = sparser network.
# verbose_level=10 prints progress per cluster
links = oracle.get_links(
    cluster_name_for_GRN_unit='leiden',
    alpha=10,
    verbose_level=10
)

  0%|          | 0/6 [00:00<?, ?it/s]

Inferring GRN for 0...


  0%|          | 0/1941 [00:00<?, ?it/s]

Inferring GRN for 1...


  0%|          | 0/1941 [00:00<?, ?it/s]

...

  0%|          | 0/1941 [00:00<?, ?it/s]

Inferring GRN for 5...


  0%|          | 0/1941 [00:00<?, ?it/s]

Inferring GRN for 6...


  0%|          | 0/1941 [00:00<?, ?it/s]

Inspect the Links object:

In [6]:
links.links_dict.keys()

dict_keys(['0', '1', '2', '3', '5', '6'])

In [8]:
links.links_dict["1"]

,source,target,coef_mean,coef_abs,p,-logp
0,Zfp282,1110051M20Rik,0.004864,0.004864,3.466755e-02,1.460077
1,Rara,1110051M20Rik,-0.002567,0.002567,2.506201e-02,1.600984
2,Zic3,1110051M20Rik,-0.001237,0.001237,1.075909e-01,0.968225
3,Rxra,1110051M20Rik,-0.001634,0.001634,1.152077e-01,0.938519
4,Pou3f2,1110051M20Rik,0.020494,0.020494,2.590321e-12,11.586646
...,...,...,...,...,...,...
719588,Foxl2,Zzef1,0.005278,0.005278,8.941157e-03,2.048606
719589,Tfcp2l1,Zzef1,-0.001355,0.001355,1.424900e-02,1.846216
719590,Foxf2,Zzef1,-0.001056,0.001056,5.484550e-01,0.260859
719591,Tfdp1,Zzef1,-0.014354,0.014354,1.359028e-05,4.866772


Filter the inferred GRNs by p-value (optional to also put a maximum of edges per cluster):

In [5]:


# Step 2: inspect raw edge distribution before filtering
links.plot_degree_distributions(
    plot_model=True,
    save=None
)

# Step 3: filter edges
# p: maximum adjusted p-value for the Ridge coefficient
# weight: rank edges by absolute coefficient value
# threshold_number: keep top N edges per cluster (not total)
links.filter_links(
    p=0.001,
    weight='coef_abs',
    threshold_number=2000
)

# Optional: check how many edges survive per cluster
links.links_dict  # dict {cluster_id: DataFrame with filtered edges}
for cluster, df in links.links_dict.items():
    print(f"Cluster {cluster}: {len(df)} edges after filtering")

# Step 4: use filtered links for simulation
oracle.get_cluster_specific_TFdict_and_TFMatrix(
    links_object=links
)
oracle.fit_GRN_for_simulation(
    alpha=10,
    use_cluster_specific_TFdict=True  # uses the filtered per-cluster networks
)

ValueError: Filtered network was not found. Prease run 'filter_links' first.

## Simulate shift

Also extracts expected expression shift after KO

In [ ]:
# =============================================================
# STEP 6: SIMULATE RMST1 KNOCKOUT
# Sets Rmst expression to 0 in all cells and propagates
# the perturbation through the fitted GRN.
# =============================================================

oracle.simulate_shift(
    perturb_condition={"Rmst": 0.0},  # KO: set Rmst to zero
    n_propagation=3   # number of propagation steps through the network
)


# =============================================================
# STEP 7: COMPUTE TRANSITION PROBABILITIES AND EMBEDDING SHIFT
# Translates the gene expression shift into a probability of
# transitioning to neighboring cells in the UMAP embedding.
# =============================================================

oracle.estimate_transition_prob(
    n_neighbors=40,
    knn_random=True,
    sampled_fraction=0.5
)

oracle.calculate_embedding_shift(sigma_corr=0.05)


# =============================================================
# STEP 8: VISUALIZE VECTOR FIELD ON UMAP
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grid-based vector field (cleaner visualization)
oracle.plot_simulation_flow_on_grid(
    scale=0.4,
    ax=axes[0],
)
axes[0].set_title('RMST1 KO — vector field (grid)')

# Single-cell arrows
oracle.plot_simulation_flow_random_sampling(
    scale=0.4,
    ax=axes[1],
    color_by='leiden',
    n_each_cluster=30
)
axes[1].set_title('RMST1 KO — vector field (single cells)')

plt.tight_layout()
#plt.savefig('../figures/rmst1_ko_vector_field.pdf', dpi=150)
plt.show()


# =============================================================
# SAVE ORACLE OBJECT FOR FURTHER ANALYSIS
# =============================================================

#oracle.to_hdf5("../data/oracle_rmst1_ko.celloracle.hdf5")